# DeePyMoD Modular PD Discovery Demo

This notebook demonstrates the new **module-aware** PD structure discovery pipeline:
- Multi-state network (R + latent states)
- Module-scoped candidate library
- Multi-equation pruning + ranking

It mirrors the flow of `demo/01_demo_report.ipynb`, but focuses on **module combinations**.

In [ ]:
# Step 0: 初始化报告运行目录
import os
from src.report.init_report_run import init_and_create_skeleton

manifest, report_md = init_and_create_skeleton(
    project_root=os.path.abspath('..'),
    run_name=None,
)

print('Run dir:', manifest['paths']['run_dir'])
print('Report :', report_md)

In [ ]:
# Step 1: 生成演示数据
import json
import numpy as np
import pandas as pd
from src.data.simulate_pkpd import generate_population_data

if 'manifest' not in globals():
    raise RuntimeError('请先执行 Step 0，确保 manifest 已初始化。')

model_name = 'IDR_INHIB_KIN_SIG'
seed = 42
n_subjects = 24

run_dir = manifest['paths']['run_dir']
data_dir = os.path.join(run_dir, 'data')
os.makedirs(data_dir, exist_ok=True)

ret = generate_population_data(
    model_name=model_name,
    seed=seed,
    n_subjects=n_subjects,
    extra_pk_iiv_sigma=0.10,
    return_pk_scale=True,
)

pop_data, subject_params, cfg, pk_scale_by_sid = ret
df = pd.DataFrame(pop_data, columns=['sid', 'time', 'C_obs', 'R_obs'])
df['sid'] = df['sid'].astype(int)

csv_path = os.path.join(data_dir, 'pkpd_long.csv')
cfg_path = os.path.join(data_dir, 'sim_config.json')
subj_path = os.path.join(data_dir, 'subject_params.json')

df.to_csv(csv_path, index=False)

def _jsonable(x):
    if isinstance(x, dict):
        return {k: _jsonable(v) for k, v in x.items()}
    if isinstance(x, list):
        return [_jsonable(v) for v in x]
    if isinstance(x, tuple):
        return tuple(_jsonable(v) for v in x)
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    return x

meta = {
    'data_source': 'simulated_from_src',
    'generator': 'src.data.simulate_pkpd.generate_population_data',
    'model_name': model_name,
    'seed': seed,
    'n_subjects': n_subjects,
    'cfg': _jsonable(cfg),
    'run_dir': run_dir,
}

with open(cfg_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
with open(subj_path, 'w', encoding='utf-8') as f:
    json.dump(_jsonable(subject_params), f, ensure_ascii=False, indent=2)

print('Data generated successfully.')
print('CSV      :', csv_path)
print('CFG      :', cfg_path)
print('SUBJ     :', subj_path)
df.head()

In [ ]:
# Step 2: 模块化结构发现（多组合）
import json
from src.pipeline.discovery import run_module_discovery
from src.configs.defaults import DEFAULTS

# 可选：自定义你想跑的模块组合（默认跑全部）
module_combos = [
    'idr',
    'idr+delay',
    'idr+tolerance',
]

results = run_module_discovery(
    pop_data=pop_data,
    active_model=model_name,
    config=DEFAULTS,
    module_combos=module_combos,
)

# 保存结果
out_path = os.path.join(manifest['paths']['run_dir'], 'tables', 'module_discovery_results.json')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2, default=str)

print('Saved:', out_path)

In [ ]:
# Step 3: 汇总每个模块组合的 Top-1 结构
import pandas as pd

rows = []
for combo, res in results.items():
    best = res['best']
    rows.append({
        'module_combo': combo,
        'k': best['k'],
        'bic_val': best['score'],
        'mse_val': best['mse_val'],
        'terms': best['terms'],
    })

summary = pd.DataFrame(rows).sort_values('bic_val')
summary